# Raggy — Llama 2-7B server on free Colab

This notebook runs **meta-llama/Llama-2-7b-chat-hf** on Colab's free GPU and gives your computer a private web address to call it.
Your Raggy backend (running in VS Code on your PC) then uses it as its language model.

**Before you start**
1. `Runtime → Change runtime type → T4 GPU → Save`
2. Left sidebar → **key icon (Secrets)** → add a secret named `HF_TOKEN` with your Hugging Face token → switch **Notebook access ON**.
3. Run the cells **top to bottom** (`Runtime → Run all` also works). Cell 5 takes several minutes the first time (13 GB download).
4. **Keep this browser tab open** while you use Raggy. If Colab disconnects, run all cells again — you will get a new URL and need to update your `.env`.

## 1. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then run again."
print("GPU:", torch.cuda.get_device_name(0), "| VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## 2. Install packages

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes fastapi "uvicorn[standard]" requests
print("Packages installed.")

## 3. Hugging Face token + server password
The token comes from your Colab **Secrets** (never typed into code). The server also gets a password so strangers who might
guess the URL cannot use your GPU. (Optional: add a Colab secret named `LLM_SERVER_API_KEY` to keep the same password every session.)

In [ ]:
import os, secrets, getpass

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Enter your Hugging Face token (input hidden): ")
os.environ["HF_TOKEN"] = HF_TOKEN

API_KEY = None
try:
    from google.colab import userdata
    API_KEY = userdata.get("LLM_SERVER_API_KEY")
except Exception:
    pass
API_KEY = API_KEY or secrets.token_urlsafe(32)
os.environ["LLM_SERVER_API_KEY"] = API_KEY
print("HF token loaded. Server password ready (shown in cell 6).")

## 4. Write the server file

In [ ]:
SERVER_CODE = r'''
import os, secrets, threading
from typing import List, Optional

import torch
from fastapi import Depends, FastAPI, Header, HTTPException
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = os.getenv("LLAMA_MODEL_ID", "meta-llama/Llama-2-7b-chat-hf")
HF_TOKEN = os.environ["HF_TOKEN"]
API_KEY = os.environ["LLM_SERVER_API_KEY"]

print(f"Loading {MODEL_ID} in 4-bit (first run downloads ~13 GB)...", flush=True)
tok = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto", token=HF_TOKEN
).eval()
print("Model ready.", flush=True)

lock = threading.Lock()  # one generation at a time on the single T4
app = FastAPI(title="Raggy Llama server (Colab)")


class Msg(BaseModel):
    role: str
    content: str


class ChatReq(BaseModel):
    messages: List[Msg]
    model: Optional[str] = None
    max_tokens: Optional[int] = 300
    temperature: Optional[float] = 0.2
    top_p: Optional[float] = 0.9


def check_key(authorization: Optional[str] = Header(default=None)) -> None:
    if not secrets.compare_digest(authorization or "", f"Bearer {API_KEY}"):
        raise HTTPException(status_code=401, detail="Unauthorized")


@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID}


@app.post("/v1/chat/completions", dependencies=[Depends(check_key)])
def chat(req: ChatReq):
    msgs = [{"role": m.role, "content": m.content} for m in req.messages]
    temp = req.temperature if req.temperature is not None else 0.2
    gen_kwargs = dict(
        max_new_tokens=max(1, min(req.max_tokens or 300, 512)),
        do_sample=temp > 0,
        top_p=req.top_p or 0.9,
        pad_token_id=tok.eos_token_id,
    )
    if temp > 0:
        gen_kwargs["temperature"] = temp
    try:
        inputs = tok.apply_chat_template(
            msgs, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
        ).to(model.device)
        with lock, torch.no_grad():
            out = model.generate(**inputs, **gen_kwargs)
        text = tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    except Exception as e:  # keep the real error in the Colab log, send a short one back
        print("Generation error:", repr(e), flush=True)
        raise HTTPException(status_code=500, detail="Generation failed")
    return {
        "object": "chat.completion",
        "choices": [{"index": 0, "message": {"role": "assistant", "content": text}, "finish_reason": "stop"}],
    }
'''

with open('server.py', 'w') as f:
    f.write(SERVER_CODE)
print('server.py written.')

## 5. Start the server and load Llama (takes a few minutes the first time)

In [ ]:
import subprocess, sys, time, requests

try:  # safe to re-run this cell
    server_proc.terminate(); server_proc.wait(timeout=10)
except Exception:
    pass

log = open("server.log", "w")
server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "server:app", "--host", "127.0.0.1", "--port", "8001"],
    stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy(),
)

start = time.time()
while True:
    if server_proc.poll() is not None:
        print(open("server.log").read()[-3000:])
        raise RuntimeError("Server stopped. Read the log above (common causes: wrong HF token, no access to the model).")
    try:
        if requests.get("http://127.0.0.1:8001/health", timeout=2).ok:
            break
    except Exception:
        pass
    if int(time.time() - start) % 30 == 0:
        print(f"  ...loading ({int(time.time() - start)}s)", open("server.log").read().strip().split("\n")[-1][:120])
    time.sleep(1)
print(f"Llama server is ready after {int(time.time() - start)}s.")

## 6. Open the tunnel and get the lines for your `.env`

In [ ]:
import re, urllib.request

if not os.path.exists("cloudflared"):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "cloudflared")
    os.chmod("cloudflared", 0o755)

try:  # safe to re-run
    tunnel_proc.terminate()
except Exception:
    pass
tunnel_log = open("tunnel.log", "w")
tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8001", "--no-autoupdate"],
                               stdout=tunnel_log, stderr=subprocess.STDOUT)

PUBLIC_URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        PUBLIC_URL = m.group(0); break
    time.sleep(1)
if not PUBLIC_URL:
    print(open("tunnel.log").read()[-2000:])
    raise RuntimeError("Could not open the tunnel. Run this cell again.")

print("=" * 70)
print("Copy these 3 lines into backend/.env on your PC, then restart the backend:\n")
print("LLM_PROVIDER=remote")
print(f"LLM_REMOTE_URL={PUBLIC_URL}/v1")
print(f"LLM_REMOTE_API_KEY={API_KEY}")
print("=" * 70)
print("Treat the password like a secret. Do not share this output.")

## 7. Test it through the public URL

In [ ]:
import time, requests
body = {"messages": [
    {"role": "system", "content": "Answer ONLY from the context. If it is not there, say you cannot find it."},
    {"role": "user", "content": "CONTEXT: The Eiffel Tower is in Paris and was completed in 1889.\n\nQUESTION: When was the Eiffel Tower completed?"},
]}
t = time.time()
r = requests.post(PUBLIC_URL + "/v1/chat/completions", json=body, headers={"Authorization": f"Bearer {API_KEY}"}, timeout=300)
print("Status:", r.status_code, f"({time.time() - t:.1f}s)")
print("Answer:", r.json()["choices"][0]["message"]["content"])
bad = requests.post(PUBLIC_URL + "/v1/chat/completions", json=body, headers={"Authorization": "Bearer wrong"}, timeout=30)
print("Wrong password is rejected:", bad.status_code == 401)

## 8. Keep running
Leave this cell running and keep the tab open while you use Raggy. Stop it (■) when you are done to free the GPU.

In [ ]:
import time
print("Server is up. Leave this cell running. Public URL:", PUBLIC_URL)
while True:
    time.sleep(600)
    alive = server_proc.poll() is None and tunnel_proc.poll() is None
    print(time.strftime("%H:%M"), "still running" if alive else "STOPPED - run cells 5 and 6 again")